## Trening Masked Language Model (MLM) BERT

### 📋 Cel notebooka
Ten notebook trenuje model BERT od zera na danych medycznych używając Masked Language Modeling (MLM).

### 🔄 Wymagania
- `corpus_combined_*.txt` - połączone dane z Synthea i MIMIC (z notebooka 03)

### 📊 Metodologia

#### 1. Tokenizacja
- **Model:** WordLevel tokenizer (każdy token oddzielony spacją = osobne słowo)
- **Uzasadnienie:** Nasze tokeny (np. `AGE_50`, `HGB_Q5`) są już "słowami" w naszym języku medycznym
- **Special tokens:** `[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]`
- **Min frequency:** 1 (chcemy znać każdy token, nawet rzadki)

#### 2. Architektura modelu
- **Baza:** BERT-base architecture (12 layers, 768 hidden size, 12 attention heads)
- **Task:** Masked Language Modeling (MLM)
- **Vocab size:** Dynamiczny (zależny od liczby unikalnych tokenów w danych)

#### 3. Parametry treningu
- **Learning rate:** 5e-5 (standardowa wartość dla pre-training BERT)
- **Batch size:** 16 (mniejszy niż fine-tuning, bo MLM jest bardziej wymagający)
- **Epochs:** 3 (wystarczające dla pre-training na domenie medycznej)
- **Max length:** 128 tokenów
- **Masking probability:** 15% (standardowa wartość dla MLM)

#### 4. Data Collator
- **Metoda:** `DataCollatorForLanguageModeling` z `mlm_probability=0.15`
- **Masking strategy:** 80% [MASK], 10% random token, 10% original token

### 📝 Output
- `medical_bert_mlm_YYYYMMDD_HHMMSS/` - wytrenowany model MLM
- Tokenizer zapisany razem z modelem

In [ ]:
import torch
from pathlib import Path
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import BertConfig, BertForMaskedLM, PreTrainedTokenizerFast
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
from torch.utils.data import Dataset
from datetime import datetime

# --- KONFIGURACJA ---
import paths_config
paths_config.ensure_dirs()
import glob
corpus_files = sorted(glob.glob(str(paths_config.DATA_PROCESSED / "corpus_combined_*.txt")), reverse=True)
if not corpus_files:
    raise FileNotFoundError("Nie znaleziono corpus_combined_*.txt w data/processed/. Uruchom najpierw notebook 03.")
CORPUS_FILE = corpus_files[0]
print(f"📂 Używam pliku: {Path(CORPUS_FILE).name}")

START_TIME = datetime.now()
OUTPUT_DIR = str(paths_config.MODELS_MLM / f"medical_bert_mlm_{START_TIME.strftime('%Y%m%d_%H%M%S')}")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Rozpoczynam setup. Device: {DEVICE}")

# --- 1. TRENING TOKENIZERA (WordLevel) ---
# Uczymy go Twojego "języka" (AGE_50, HGB_Q1_CRITICAL itp.)
print("🔨 Trenowanie tokenizera...")

# Używamy modelu WordLevel - każdy token oddzielony spacją to osobne słowo
tokenizer_raw = Tokenizer(models.WordLevel(unk_token="[UNK]"))
tokenizer_raw.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.WordLevelTrainer(
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
    min_frequency=1  # Chcemy znać każdy, nawet rzadki token
)

# Trenujemy na pliku tekstowym
tokenizer_raw.train([CORPUS_FILE], trainer)

# Opakowujemy w klasę kompatybilną z Transformers
tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer_raw)
tokenizer.pad_token = "[PAD]"
tokenizer.unk_token = "[UNK]"
tokenizer.cls_token = "[CLS]"
tokenizer.sep_token = "[SEP]"
tokenizer.mask_token = "[MASK]"

vocab_size = tokenizer.vocab_size
print(f"✅ Tokenizer gotowy! Rozmiar słownika: {vocab_size}")
# Zapiszmy tokenizer od razu
tokenizer.save_pretrained(OUTPUT_DIR)

# --- 2. DATASET I COLLATOR ---
class LineByLineTextDataset(Dataset):
    def __init__(self, tokenizer, file_path, block_size=128):
        self.examples = []
        with open(file_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]
        
        print(f"📄 Wczytywanie {len(lines)} linii do datasetu...")
        
        # Tokenizacja wsadowa (batch) dla szybkości
        batch_encoding = tokenizer(
            lines, 
            add_special_tokens=True, 
            truncation=True, 
            max_length=block_size,
            padding="max_length",
            return_tensors="pt"
        )
        
        self.input_ids = batch_encoding["input_ids"]
        self.attention_mask = batch_encoding["attention_mask"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        return {
            "input_ids": self.input_ids[i],
            "attention_mask": self.attention_mask[i],
            # Labels są tworzone automatycznie przez DataCollator
        }

dataset = LineByLineTextDataset(tokenizer, CORPUS_FILE, block_size=128)

# Data Collator automatycznie maskuje 15% tokenów (MLM)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=True, 
    mlm_probability=0.15
)

# --- 3. MODEL (Small Medical BERT) ---
config = BertConfig(
    vocab_size=vocab_size,
    hidden_size=256,
    num_hidden_layers=6,   # Liczba warstw
    num_attention_heads=8,
    intermediate_size=2048,
    max_position_embeddings=128, # Długość sekwencji
    type_vocab_size=1,
)

model = BertForMaskedLM(config)
print(f"🧠 Model zainicjowany. Parametry: {model.num_parameters():,}")
from transformers import EarlyStoppingCallback
# --- 4. TRENING (Trainer API) ---
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=64,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    learning_rate=1e-4,
    weight_decay=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    logging_steps=100,
    report_to="none"
)

from torch.utils.data import Subset

# Wybieramy np. 2000 pierwszych przykładów z datasetu
eval_subset_size = 2000
eval_dataset = Subset(dataset, range(min(eval_subset_size, len(dataset))))


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)


print("🚀 Start Pre-training (MLM)...")
trainer.train()

# --- 5. ZAPIS FINALNY ---
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"💾 Model zapisany w: {OUTPUT_DIR}")